# Windows server setup + train (trần 15h, snapshot giờ 9)

Chạy trong conda env `aiml` (`conda activate aiml`) rồi chạy từng cell theo thứ tự.
Cell cài đặt tự kiểm tra torch CUDA >= 2.6; PyPI chỉ có bản CPU cho Windows nên
cell sẽ cài từ index CUDA của PyTorch trước khi cài requirements.
Dừng ngay khi cell kiểm tra FAIL, không chạy train.
Dùng Pikafish bản Windows: `pikafish/Pikafish-Windows-x86-64-universal.exe`.
Config Windows dùng 8 self-play worker và checkpoint thưa hơn để tận dụng
RTX 3090 24GB + Xeon 25 core và giảm I/O.
Train dừng ở 14h30, sau đó đo cuối trong buffer 30 phút để kết thúc khoảng 15h.
Không mount Google Drive. Cuối giờ copy `models/snapshot_9h.pt` và `models/best_model.pt` về máy.


In [ ]:
!nvidia-smi
!python --version
!pip --version

In [ ]:
import subprocess
import sys

# PyPI torch tren Windows la ban CPU; cai ban CUDA tu index cua PyTorch.
# Doi cu124 thanh cu121/cu126 neu driver cua server khac.
CUDA_INDEX = 'https://download.pytorch.org/whl/cu124'
PROBE = """
import sys
import torch
version = tuple(int(part) for part in torch.__version__.split('+')[0].split('.')[:2])
ready = bool(torch.version.cuda) and torch.cuda.is_available() and version >= (2, 6)
sys.exit(0 if ready else 1)
"""
probe = subprocess.run([sys.executable, '-c', PROBE], capture_output=True, text=True)
if probe.returncode != 0:
    print('CUDA torch >= 2.6 not available; installing from', CUDA_INDEX)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--force-reinstall',
                    'torch', '--index-url', CUDA_INDEX], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
VERIFY = """
import torch
assert torch.version.cuda and torch.cuda.is_available(), 'CUDA torch was replaced or is unavailable'
version = tuple(int(part) for part in torch.__version__.split('+')[0].split('.')[:2])
assert version >= (2, 6), f'torch >= 2.6 required, got {torch.__version__}'
print(torch.__version__, torch.version.cuda)
"""
subprocess.run([sys.executable, '-c', VERIFY], check=True)


In [ ]:
import os
import sys
from pathlib import Path

import torch

print('python:', sys.version.split()[0])
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for the 15h training run')
props = torch.cuda.get_device_properties(0)
print('gpu:', props.name, f'| vram: {props.total_memory / 1024 ** 3:.1f}GB')
print(f'cpu: {os.cpu_count()}')
engine = Path('pikafish/Pikafish-Windows-x86-64-universal.exe')
required = [
    'pikafish/Pikafish-Windows-x86-64-universal.exe',
    'pikafish/pikafish.nnue',
    'configs/train_depth2_16h_windows.yaml',
    'scripts/run_16h.py',
    'scripts/train.py',
    'scripts/build_teacher_labels.py',
    'scripts/pretrain_supervised.py',
    'scripts/measure_vs_pikafish.py',
]
missing = [path for path in required if not Path(path).is_file()]
for path in required:
    print(('OK  ' if Path(path).is_file() else 'MISS'), path)
if missing:
    raise FileNotFoundError('Missing required files: ' + ', '.join(missing))


In [ ]:
!python tests/test_game.py
!python -m pytest tests/test_engine.py tests/test_search_score.py tests/test_teacher_label.py tests/test_worker_select.py tests/test_sparring_game.py -q

In [ ]:
# Trần 15h. Giờ 9: snapshot models/snapshot_9h.pt. Giờ 15: dừng hẳn.
!python scripts/run_16h.py --config configs/train_depth2_16h_windows.yaml
